<a href="https://colab.research.google.com/github/syntizen/Union-Chess/blob/main/Union_Chess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import random
import time
from ipywidgets import HTML
from IPython.display import display

class ChessPiece:
    def __init__(self, name, color):
        self.name = name
        self.color = color
        self.symbol = self.assign_traditional_symbol()

    def assign_traditional_symbol(self):
        symbols = {
            "King": "♚", "Queen": "♛", "Rook": "♜",
            "Bishop": "♝", "Knight": "♞", "Pawn": "♟"
        }
        return symbols.get(self.name, "?")

    def __str__(self):
        return self.symbol

class ChessGame:
    def __init__(self):
        self.rows = 16
        self.cols = 24
        self.board = self.create_board()
        self.current_turn = "White"
        self.moves_left_this_turn = 3
        self.moved_pieces_this_turn = set()

        self.animating_piece = None
        self.animating_pos = None
        self.position_history = {"White": [], "Black": []}

    def create_board(self):
        board = [[None for _ in range(self.cols)] for _ in range(self.rows)]

        # --- WHITE INITIALIZATION ---
        white_pieces_order = ["Rook", "Knight", "Bishop", "Queen", "King", "Bishop", "Knight", "Rook"] * 3
        for i in range(self.cols):
            board[0][i] = ChessPiece(white_pieces_order[i], "White")
            board[1][i] = ChessPiece("Pawn", "White")
            board[2][i] = ChessPiece("Pawn", "White")

        # --- BLACK INITIALIZATION ---
        black_pieces_order = list(reversed(white_pieces_order))
        for i in range(self.cols):
            board[15][i] = ChessPiece(black_pieces_order[i], "Black")
            board[14][i] = ChessPiece("Pawn", "Black")
            board[13][i] = ChessPiece("Pawn", "Black")

        return board

    def get_valid_moves(self, r, c, ignore_turn=False, ignore_moved_restriction=False):
        piece = self.board[r][c]
        if not piece: return []
        if not ignore_turn and piece.color != self.current_turn: return []
        if not ignore_turn and not ignore_moved_restriction and (r, c) in self.moved_pieces_this_turn: return []

        moves = []
        name = piece.name

        # --- ADVANCED DOUBLE-LAYER PAWN ENGINE ---
        if name == "Pawn":
            direction = 1 if piece.color == "White" else -1
            layer_1_start = 1 if piece.color == "White" else 14
            layer_2_start = 2 if piece.color == "White" else 13

            if 0 <= r + direction < self.rows and self.board[r + direction][c] is None:
                moves.append((r + direction, c))
                if r == layer_1_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))
                        if self.board[r + 3 * direction][c] is None:
                            moves.append((r + 3 * direction, c))
                elif r == layer_2_start:
                    if self.board[r + 2 * direction][c] is None:
                        moves.append((r + 2 * direction, c))

            for dc in [-1, 1]:
                target_c = c + dc
                target_r = r + direction
                if 0 <= target_r < self.rows and 0 <= target_c < self.cols:
                    target = self.board[target_r][target_c]
                    if target and target.color != piece.color:
                        moves.append((target_r, target_c))

        # --- KNIGHT MOVES ---
        elif name == "Knight":
            offsets = [(-2,-1), (-2,1), (-1,-2), (-1,2), (1,-2), (1,2), (2,-1), (2,1)]
            for dr, dc in offsets:
                tr, tc = r + dr, c + dc
                if 0 <= tr < self.rows and 0 <= tc < self.cols:
                    if self.board[tr][tc] is None or self.board[tr][tc].color != piece.color:
                        moves.append((tr, tc))

        # --- SLIDING PIECES ---
        else:
            directions = []
            max_step = max(self.rows, self.cols)
            if name in ["Rook", "Queen"]:
                directions.extend([(-1,0), (1,0), (0,-1), (0,1)])
            if name in ["Bishop", "Queen"]:
                directions.extend([(-1,-1), (-1,1), (1,-1), (1,1)])
            if name == "King":
                directions.extend([(-1,0), (1,0), (0,-1), (0,1), (-1,-1), (-1,1), (1,-1), (1,1)])
                max_step = 1

            for dr, dc in directions:
                for step in range(1, max_step + 1):
                    tr, tc = r + dr * step, c + dc * step
                    if not (0 <= tr < self.rows and 0 <= tc < self.cols):
                        break
                    target = self.board[tr][tc]
                    if target is None:
                        moves.append((tr, tc))
                    elif target.color != piece.color:
                        moves.append((tr, tc))
                        break
                    else:
                        break
        return moves

    def get_all_legal_moves(self, color):
        all_moves = []
        for r in range(self.rows):
            for c in range(self.cols):
                piece = self.board[r][c]
                if piece and piece.color == color:
                    valid_to_squares = self.get_valid_moves(r, c)
                    for tr, tc in valid_to_squares:
                        all_moves.append(((r, c), (tr, tc)))
        return all_moves

    def calculate_path_points(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]
        if not piece: return [(fr, fc), (tr, tc)]
        if piece.name == "Knight": return [(fr, fc), (tr, fc), (tr, tc)]
        dr, dc = tr - fr, tc - fc
        steps = max(abs(dr), abs(dc))
        if steps == 0: return [(fr, fc)]
        return [(fr + (dr * i) // steps, fc + (dc * i) // steps) for i in range(steps + 1)]

    def count_kings(self):
        counts = {"White": 0, "Black": 0}
        for row in self.board:
            for piece in row:
                if piece and piece.name == "King":
                    counts[piece.color] += 1
        return counts

    def finalize_move_data(self, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq
        piece = self.board[fr][fc]

        if piece.name == "Pawn" and (tr == 15 or tr == 0):
            piece = ChessPiece("Queen", piece.color)

        self.board[tr][tc] = piece
        self.board[fr][fc] = None

        self.position_history[piece.color].append(to_sq)
        if len(self.position_history[piece.color]) > 6:
            self.position_history[piece.color].pop(0)

        self.moved_pieces_this_turn.add((tr, tc))
        self.moves_left_this_turn -= 1

        if self.moves_left_this_turn == 0:
            self.current_turn = "Black" if self.current_turn == "White" else "White"
            self.moves_left_this_turn = 3
            self.moved_pieces_this_turn.clear()

# --- THE HIGH-IQ THREAT-AWARE AI ROBOT ---
class ChessRobot:
    def __init__(self, color):
        self.color = color
        self.enemy_color = "Black" if color == "White" else "White"
        self.piece_values = {"Pawn": 15, "Knight": 40, "Bishop": 45, "Rook": 65, "Queen": 120, "King": 9999}

    def evaluate_move_inplace(self, game, from_sq, to_sq):
        fr, fc = from_sq
        tr, tc = to_sq

        moving_piece = game.board[fr][fc]
        target_piece = game.board[tr][tc]

        score_delta = 0

        # 1. Capture Value Weighting
        if target_piece:
            score_delta += self.piece_values.get(target_piece.name, 0) * 20

        # 2. Positional Progression Values
        direction = 1 if self.color == "White" else -1
        rank_advance = tr - fr
        score_delta += (rank_advance * direction) * 2.5

        # Avoid basic repetitions
        if to_sq in game.position_history[self.color]:
            score_delta -= 15.0

        # 3. HIGH-INTELLECT THREAT DETECTION MATRIX
        # We simulate the threat profile on the grid *before* finalizing
        enemy_attacks_target = False
        friendly_guards_target = False

        # Scan everything looking at the target square
        for r in range(game.rows):
            for c in range(game.cols):
                # Don't look from the piece that is currently moving away
                if (r, c) == (fr, fc): continue

                p = game.board[r][c]
                if p:
                    # Is an enemy piece threatening our landing spot?
                    if p.color == self.enemy_color:
                        # Quick check: if the target square is in their attack vector array
                        valid_enemy_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_enemy_moves:
                            enemy_attacks_target = True
                    # Is a friendly piece guarding our landing spot?
                    elif p.color == self.color:
                        valid_friendly_moves = game.get_valid_moves(r, c, ignore_turn=True, ignore_moved_restriction=True)
                        if (tr, tc) in valid_friendly_moves:
                            friendly_guards_target = True

        # ANTI-SUICIDE LOGIC GAUNTLET:
        if enemy_attacks_target:
            my_value = self.piece_values.get(moving_piece.name, 0)
            if not friendly_guards_target:
                # Completely unprotected suicide dive -> Apply massive penalty
                score_delta -= my_value * 50
            else:
                # Protected square, but if it's an expensive piece (like a Queen) trading for a cheap square, penalize it
                if moving_piece.name in ["Queen", "Rook", "Bishop", "Knight"]:
                    score_delta -= my_value * 15

        # 4. Phalanx Shield Formations (Pawn structures remain smart)
        if moving_piece.name == "Pawn":
            for dc in [-1, 1]:
                if 0 <= tc + dc < game.cols:
                    adj_p = game.board[tr][tc + dc]
                    if adj_p and adj_p.name == "Pawn" and adj_p.color == self.color:
                        score_delta += 6.0
            support_row = tr - direction
            for dc in [-1, 1]:
                if 0 <= support_row < game.rows and 0 <= tc + dc < game.cols:
                    guard_p = game.board[support_row][tc + dc]
                    if guard_p and guard_p.name == "Pawn" and guard_p.color == self.color:
                        score_delta += 5.0

        # 5. Proximity Tracking (King Defense vs King Assassination)
        for r in range(game.rows):
            for c in range(game.cols):
                p = game.board[r][c]
                if p and p.name == "King":
                    if p.color == self.color:
                        old_k_dist = max(abs(fr - r), abs(fc - c))
                        new_k_dist = max(abs(tr - r), abs(tc - c))
                        if moving_piece.name == "Pawn" and new_k_dist > 3 and old_k_dist <= 3:
                            score_delta -= 25.0
                    elif p.color == self.enemy_color:
                        old_dist = abs(fr - r) + abs(fc - c)
                        new_dist = abs(tr - r) + abs(tc - c)
                        if new_dist < old_dist:
                            score_delta += (40 - new_dist) * 2.5

        return score_delta

    def select_best_move(self, game):
        legal_moves = game.get_all_legal_moves(self.color)
        if not legal_moves: return None

        best_score = -float('inf')
        best_choices = []

        for move in legal_moves:
            score = self.evaluate_move_inplace(game, move[0], move[1])
            if score > best_score:
                best_score = score
                best_choices = [move]
            elif score == best_score:
                best_choices.append(move)

        if best_choices:
            return random.choice(best_choices)
        return random.choice(legal_moves)

def generate_board_html(game, current_logs=[], turn_counter=0):
    kings = game.count_kings()
    logs_html = "".join([f"<div style='margin-top:2px;'>• {log}</div>" for log in current_logs])
    if not logs_html: logs_html = "Processing Matrix..."

    html = f"""
    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; border-radius: 8px; width: 820px; margin-bottom: 10px; box-sizing: border-box; border: 1px solid #ff4a4a;">
        <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; color: #ff4a4a;">
            🧠 HIGH-IQ THREAT DEFLECTION INTELLIGENCE GRID &nbsp;|&nbsp; Global Match Turn: {turn_counter}
        </div>
        <div style="font-size: 13px; margin-bottom: 8px;">
            <span style="color: #fff; font-weight:bold;">🤍 White Monarch Files: {kings['White']}</span> &nbsp;&nbsp;&nbsp;&nbsp;
            <span style="color: #aaa; font-weight:bold;">🖤 Black Monarch Files: {kings['Black']}</span>
        </div>
        <div style="background: #222; padding: 8px 12px; border-left: 4px solid #ff4a4a; font-size: 12px; color: #ddd; line-height: 1.4; height: 54px; overflow-y: auto; box-sizing: border-box;">
            <b>Threat Assessment Stream Logs:</b>
            {logs_html}
        </div>
    </div>
    <table style="border-collapse: collapse; border: 3px solid #333; font-family: sans-serif;">
    """
    for r in range(game.rows - 1, -1, -1):
        html += "<tr>"
        for c in range(game.cols):
            is_light = (r + c) % 2 != 0
            bg_color = "#f0d9b5" if is_light else "#b58863"

            piece = game.board[r][c]
            piece_html = ""
            active_render_piece = piece
            if game.animating_piece and game.animating_pos == (r, c):
                active_render_piece = game.animating_piece

            if active_render_piece:
                circle_bg = "#ffffff" if active_render_piece.color == "White" else "#222222"
                glyph_color = "#151515" if active_render_piece.color == "White" else "#ffffff"
                stroke = "1px solid #999" if active_render_piece.color == "White" else "1px solid #444"

                piece_html = f"""
                <div style="width: 26px; height: 26px; line-height: 25px; border-radius: 50%;
                            background-color: {circle_bg}; color: {glyph_color}; font-size: 18px;
                            text-align: center; margin: auto; border: {stroke};
                            box-shadow: 1px 1px 3px rgba(0,0,0,0.4); user-select: none; transition: all 0.03s ease;">
                    {str(active_render_piece)}
                </div>
                """
            html += f"""<td style="width: 32px; height: 32px; background-color: {bg_color}; text-align: center; vertical-align: middle; padding: 0; border: 1px solid rgba(0,0,0,0.05);">{piece_html}</td>"""
        html += "</tr>"
    html += "</table>"
    return html

In [2]:
# Initialize Game Matrix
game = ChessGame()
white_bot = ChessRobot("White")
black_bot = ChessRobot("Black")

board_widget = HTML(value=generate_board_html(game, ["Threat analyzer matrices running calibration vectors."], 0))
display(board_widget)

turn_counter = 0

while True:
    king_counts = game.count_kings()
    if king_counts["White"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - BLACK WINS! Threat intelligence domination victory."], turn_counter)
        break
    if king_counts["Black"] == 0:
        board_widget.value = generate_board_html(game, ["🏆 GAME OVER - WHITE WINS! Threat intelligence domination victory."], turn_counter)
        break

    active_color = game.current_turn
    bot = white_bot if active_color == "White" else black_bot

    turn_summary_logs = []
    actions_taken = 0

    for step in range(3):
        if game.count_kings()["White" if active_color == "Black" else "Black"] == 0:
            break

        chosen_move = bot.select_best_move(game)
        if chosen_move:
            from_sq, to_sq = chosen_move
            moving_piece = game.board[from_sq[0]][from_sq[1]]
            target_square = game.board[to_sq[0]][to_sq[1]]

            promo_note = " -> ♕ PROMOTED" if moving_piece.name == "Pawn" and (to_sq[0] == 15 or to_sq[0] == 0) else ""
            capture_note = f" (Captured {target_square.name}!)" if target_square else ""

            log_line = f"Action {step+1}: Verified {moving_piece.color} {moving_piece.name} safely to {to_sq}{promo_note}{capture_note}"
            turn_summary_logs.append(log_line)

            # --- HIGH SPEED VECTOR ANIMATION ---
            path_frames = game.calculate_path_points(from_sq, to_sq)
            game.animating_piece = moving_piece
            game.board[from_sq[0]][from_sq[1]] = None

            for frame_pos in path_frames:
                game.animating_pos = frame_pos
                board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
                time.sleep(0.012)

            game.animating_piece = None
            game.animating_pos = None

            game.board[from_sq[0]][from_sq[1]] = moving_piece
            game.finalize_move_data(from_sq, to_sq)
            actions_taken += 1
        else:
            break

    if actions_taken == 0:
        game.current_turn = "Black" if game.current_turn == "White" else "White"
        game.moves_left_this_turn = 3
        game.moved_pieces_this_turn.clear()

    turn_counter += 1
    board_widget.value = generate_board_html(game, turn_summary_logs, turn_counter)
    time.sleep(0.12)

HTML(value='\n    <div style="font-family: monospace; background-color: #111; color: #fff; padding: 15px; bord…